# 10 Spatial statistics of M05 residuals

Question (T109 only): are the errors of the approved M05 drought outlook (ridge, D43) spatially
clustered across Karnataka districts? Moran's I per test month on row-standardised queen
contiguity, 999 permutations, Benjamini-Hochberg across months, and a training-only
semivariogram. Residual = observed SPI-3 minus the ridge forecast (SPI units).

- Moran's I: the saved v1 ridge (never refitted) on the locked test, 60 issue months 2020-01 to
  2024-12, 30 S17 districts (geoBoundaries 2021 lines). Pseudo p is two-sided.
- Semivariogram: out-of-sample residuals of ridge refits at origins 2000, 2005, 2010 and 2015
  (the D97 rolling scheme, v1 alpha and features, 92-day purge), target years 2000-2019 only.
  No 2020+ row enters it.
- M04 (gauge gap filling) is blocked (`reports/models/m04.md`): there are no M04 outputs, so the
  plan's M04 comparison cannot be made. Nothing is substituted.

Frozen inputs: S08 IMD year files 1981-2024 named in the raw ledger (local only: IMD forbids
reproduction, so only derived statistics are exported), S17 evidence e05/e06
(`tests/fixtures/s17/manifest.json`) and the saved v1 fit (`data/models/m05/m05_fit.json`,
checked against the release dataset hash). This is a hindcast (D39).

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/10_spatial_statistics/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import math
import os
import textwrap
from datetime import date
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import shapely
from pyproj import Transformer

from atlas.analysis.quality import caption
from atlas.analysis.spatial import (
    benjamini_hochberg,
    permutation_test,
    queen_neighbours,
    row_standardised,
    semivariogram,
)
from atlas.config import load_state
from atlas.contracts import RawArtifact
from atlas.features import m05 as features
from atlas.ingest.s17 import S17
from atlas.models import m05

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/10_spatial_statistics"))
OUT.mkdir(parents=True, exist_ok=True)
RAW = Path("data/raw")
SEED, PERMUTATIONS, Q = 109, 999, 0.05
TEST = (date(2020, 1, 1), date(2024, 12, 31))
BIN_KM = 50


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)


def s17(eid):
    manifest = json.loads(Path("tests/fixtures/s17/manifest.json").read_text("utf-8"))
    e = next(x for x in manifest["evidence"] if x["id"] == eid)
    return RawArtifact(
        sha256=e["sha256"],
        path=RAW / e["raw_path"],
        fetched_at=e["fetched_at"],
        media_type=e["media_type"],
        response_status=e["status"],
        source_url=e["url"],
        schema_mapping_version="s17-v1",
    )

In [ ]:
ka = load_state(Path("configs/states/ka.yaml"))
e06 = s17("e06")
districts = S17("ka").parse(e06, ka.source("S17"), adm1=s17("e05"))
raws = features.year_artifacts(RAW, range(1981, 2025))
rows = features.feature_rows(features.spi_table(features.district_monthly_rain(raws, districts)))
fit = json.loads(Path("data/models/m05/m05_fit.json").read_text("utf-8"))
release = json.loads(Path("models/registry/M05/v1.json").read_text("utf-8"))
assert m05.table_sha256(rows) == fit["dataset_sha256"] == release["dataset_sha256"]
test = m05.rows_in(rows, *TEST)
mean, std = (np.asarray(fit["scaler"][k]) for k in ("mean", "std"))
forecast = m05._design(test, mean, std)[0] @ np.asarray(fit["ridge"]["coef"])
residual = np.asarray(test[m05.TARGET].to_pylist(), dtype=float) - (
    forecast + fit["ridge"]["intercept"]
)
known_before_target = all(
    k < t
    for k, t in zip(test["known_until"].to_pylist(), test["target_month"].to_pylist(), strict=True)
)
fetched = sorted(r.fetched_at for r in raws)
print(test.num_rows, "test district-months; MAE", float(np.abs(residual).mean()))

In [ ]:
place = districts["place_id"].to_pylist()
names = dict(zip(place, districts["name"].to_pylist(), strict=True))
polys = [shapely.from_wkb(b) for b in districts["geometry"].to_pylist()]
neighbours, islands = queen_neighbours(polys)
w = row_standardised(neighbours)
to_km = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True).transform
xy = np.array(
    [
        [c.x / 1000, c.y / 1000]
        for c in (shapely.transform(p, to_km, interleaved=False).centroid for p in polys)
    ]
)
links = sum(len(n) for n in neighbours)
write_csv(
    "district_neighbours.csv",
    [
        {
            "place_id": p,
            "name": names[p],
            "neighbours": len(n),
            "neighbour_names": "; ".join(sorted(names[place[j]] for j in n)),
        }
        for p, n in zip(place, neighbours, strict=True)
    ],
)
print(len(place), "districts;", links // 2, "shared borders;", len(islands), "islands")

In [ ]:
rng = np.random.default_rng(SEED)
col = {p: i for i, p in enumerate(place)}
months = sorted(set(test["target_month"].to_pylist()))
moran = []
for m in months:
    x = np.full(len(place), np.nan)
    for p, t, r in zip(
        test["place_id"].to_pylist(), test["target_month"].to_pylist(), residual, strict=True
    ):
        if t == m:
            x[col[p]] = r
    if np.isnan(x).any():  # never filled: a month with a missing district is not tested
        moran.append({"month": m.isoformat()[:7], "districts": int((~np.isnan(x)).sum())})
        continue
    t = permutation_test(x, w, rng, PERMUTATIONS)
    moran.append({"month": m.isoformat()[:7], "districts": len(x)} | t)
tested = [r for r in moran if "p" in r]
for r, q in zip(tested, benjamini_hochberg([r["p"] for r in tested]), strict=True):
    r["q_bh"], r["significant_q05"] = float(q), bool(q <= Q)
write_csv("moran_by_month.csv", tested)
sig = [r for r in tested if r["significant_q05"]]
raw_sig = [r for r in tested if r["p"] <= Q]
print(len(tested), "of", len(months), "months tested;", len(raw_sig), "with p <= 0.05;")
print(len(sig), "significant after BH;", sum(r["moran_i"] > r["expected"] for r in sig), "positive")

In [ ]:
res = m05.rolling_residuals(rows, fit["ridge"]["alpha"])
assert res["year"].max() < 2020, "variogram must use training-period residuals only"
times = sorted({(int(y), int(m)) for y, m in zip(res["year"], res["month"], strict=True)})
row_of = {t: i for i, t in enumerate(times)}
grid = np.full((len(times), len(place)), np.nan)
for y, m, p, r in zip(res["year"], res["month"], res["place_id"], res["ridge"], strict=True):
    grid[row_of[int(y), int(m)], col[p]] = r
far = max(math.dist(a, b) for a in xy for b in xy)
edges = list(range(0, BIN_KM * math.ceil(far / BIN_KM) + 1, BIN_KM))
vario = semivariogram(xy, grid, edges)
overall = semivariogram(xy, grid, [0, edges[-1]])[0]
write_csv("variogram_training.csv", vario)
for v in vario:
    print(v["lo_km"], v["hi_km"], v["site_pairs"], v["pair_times"], v["gamma"])

In [ ]:
cap_moran = caption(
    "M05 v1 ridge residuals on S08 IMD (S17 geoBoundaries 2021 districts, queen weights)",
    "Moran's I of observed minus forecast SPI-3 per issue month, 999 permutations",
    "dimensionless; residual in SPI units",
    tested[0]["month"],
    tested[-1]["month"],
    f"{len(tested) * len(place)} of {len(months) * len(place)} district-months, "
    f"{len(tested)} of {len(months)} months",
)
fig, ax = plt.subplots(figsize=(8, 4))
xs = range(len(tested))
ax.axhline(tested[0]["expected"], color="grey", lw=0.8, label="E[I] = -1/29")
ax.plot(xs, [r["moran_i"] for r in tested], "o", ms=3, color="tab:blue", label="q > 0.05")
ax.plot(
    [i for i, r in enumerate(tested) if r["significant_q05"]],
    [r["moran_i"] for r in sig],
    "o",
    ms=5,
    color="tab:red",
    label="BH q <= 0.05",
)
ax.set_xticks(range(0, len(tested), 12), [r["month"] for r in tested[::12]])
ax.set_ylabel("Moran's I")
ax.legend(fontsize=7)
ax.set_title("M05 residual spatial autocorrelation, locked test", fontsize=10)
fig.subplots_adjust(bottom=0.22)
fig.text(0.01, 0.01, textwrap.fill(cap_moran, 160), fontsize=6, va="bottom")
fig.savefig(OUT / "moran_by_month.png", dpi=80, metadata={"Software": None})
plt.close(fig)

cap_vario = caption(
    "M05 rolling-origin ridge refits on S08 IMD (S17 district centroids, EPSG:6933)",
    "semivariance of same-month residual differences by centroid distance",
    "SPI units squared; distance km",
    f"{times[0][0]}-{times[0][1]:02d}",
    f"{times[-1][0]}-{times[-1][1]:02d}",
    f"{overall['pair_times']} of {overall['site_pairs'] * overall['times']} pair-months, "
    f"{int((~np.isnan(grid)).sum())} of {grid.size} district-months",
)
used = [v for v in vario if v["gamma"] is not None]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([v["mean_km"] for v in used], [v["gamma"] for v in used], "o-")
for v in used:
    ax.annotate(
        str(v["site_pairs"]),
        (v["mean_km"], v["gamma"]),
        fontsize=6,
        xytext=(2, 3),
        textcoords="offset points",
    )
ax.axhline(overall["gamma"], color="grey", lw=0.8, label="all pairs")
ax.set_xlabel("Centroid distance (km); labels = district pairs per bin")
ax.set_ylabel("Semivariance (SPI^2)")
ax.set_ylim(bottom=0)
ax.legend(fontsize=7)
ax.set_title("Training-only residual semivariogram (2000-2019)", fontsize=10)
fig.subplots_adjust(bottom=0.25)
fig.text(0.01, 0.01, textwrap.fill(cap_vario, 140), fontsize=6, va="bottom")
fig.savefig(OUT / "variogram_training.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
summary = {
    "model": {
        "id": "M05",
        "version": "v1",
        "champion": fit["choice"],
        "dataset_sha256": release["dataset_sha256"],
        "train_cutoff": release["cutoffs"]["train"],
        "evidence": "hindcast (D39)",
        "features_known_before_target": known_before_target,
    },
    "weights": {
        "rule": "queen (shared boundary point), row-standardised",
        "districts": len(place),
        "shared_borders": links // 2,
        "islands": [names[place[i]] for i in islands],
        "min_neighbours": min(len(n) for n in neighbours),
        "max_neighbours": max(len(n) for n in neighbours),
    },
    "moran": {
        "residuals": "observed minus saved v1 ridge forecast (target_spi3, SPI units)",
        "months": len(months),
        "months_tested": len(tested),
        "district_months": len(tested) * len(place),
        "district_months_expected": len(months) * len(place),
        "first_month": tested[0]["month"],
        "last_month": tested[-1]["month"],
        "permutations": PERMUTATIONS,
        "seed": SEED,
        "p_value": "two-sided: (1 + #{|I_perm - E| >= |I_obs - E|}) / (999 + 1), E = -1/(N-1)",
        "bh_family": "all tested months",
        "q": Q,
        "significant_raw_p05": len(raw_sig),
        "significant_bh": len(sig),
        "significant_bh_positive": sum(r["moran_i"] > r["expected"] for r in sig),
        "moran_i_min": min(r["moran_i"] for r in tested),
        "moran_i_median": float(np.median([r["moran_i"] for r in tested])),
        "moran_i_max": max(r["moran_i"] for r in tested),
        "caption": cap_moran,
    },
    "variogram": {
        "residuals": "ridge refits at origins 2000, 2005, 2010, 2015 (v1 alpha, 92-day purge)",
        "years": sorted({int(y) for y in res["year"]}),
        "months": len(times),
        "district_months": int((~np.isnan(grid)).sum()),
        "district_months_expected": int(grid.size),
        "distance": "district centroid in EPSG:6933, km",
        "bin_km": BIN_KM,
        "site_pairs": overall["site_pairs"],
        "pair_months": overall["pair_times"],
        "semivariance_all_pairs": overall["gamma"],
        "caption": cap_vario,
    },
    "inputs": {
        "s08_years": [1981, 2024],
        "s08_files": len(raws),
        "s08_first_fetch": fetched[0].isoformat(),
        "s08_last_fetch": fetched[-1].isoformat(),
        "s17_fetch": e06.fetched_at.isoformat(),
        "s17_sha256": e06.sha256,
    },
    "limits": [
        "M04 is blocked (no gauge-day data, D113): no M04 outputs exist, so no comparison is made.",
        "30 S17 districts only (geoBoundaries 2021; Ballari includes Vijayanagara); "
        "neighbours outside Karnataka are not in the weights.",
        "60 test months: BH at q = 0.05 controls the false discovery rate over months, "
        "not the spatial structure within a month.",
        "Variogram residuals come from refits, not the champion itself "
        "(trained to 2016-09); test years are excluded by design.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))